# Explore the metagenome accessions missing some or all of the core microbes

In [1]:
import polars as pl
from io import StringIO

In [2]:
CORE_NAMES = [ x.strip() for x in open('../inputs.branchwater/names.list') ]

## Load in the metadata

In [3]:
metadata_df = (
    pl.scan_parquet("/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet")
    .filter(pl.col("acc") != "NP")
    .filter(pl.col("assay_type") == "WGS")
    .collect()
)
metadata_df.head(1)

sample_name,sample_name_sam,acc,assay_type,avgspotlen,bioproject,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,list[str],str,str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""27""",[],"""SRR28523869""","""WGS""",5336,"""PRJNA1095378""","""SAMN40717046""","[""MIMARKS.survey"", ""MIGS/MIMS/MIMARKS.human-associated""]","""UNIVERSITY OF FLORIDA""",2019-01-21,"""public""","[""fastq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX24124678""","""USA""","""North America""","[""USA""]",null,"""MinION""","""27""","""SINGLE""","""RANDOM""","""METAGENOMIC""",null,258,221,"""human metagenome""","""OXFORD_NANOPORE""",2024-04-02 00:00:00 UTC,"""SRS20910198""","""SRP499205""",null,null,null,null,"""[""2019-01-21""]""",null,null,"""[""human stool""]""",null,"""[""preterm neonate stool""]""",null,"""[""not applicable""]""",null,"""[""Homo sapiens""]""",null,null,null,null,null,null,null,null,null,"""""29.6520 N 82.3250 W""""",null,null,null,null,"""258178589""","""232696690""","""""2024-04-02T09:18:00.000Z""""",null,"""""1095378"""""


## Load the manysearch results and filter at our default threshold of 20

In [4]:
df = (
    pl.scan_parquet('../outputs.cds3/pq/mag+gtdb.cds3.x.3216.manysearch.parquet')
    .filter(pl.col('intersect_hashes') >= 20)
    .with_columns(
        species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "),
        acc=pl.col('match_name'),
    )
).collect()

In [5]:
df.head(1)

query_name,query_md5,match_name,containment,intersect_hashes,ksize,scaled,moltype,match_md5,jaccard,max_containment,average_abund,median_abund,std_abund,query_containment_ani,match_containment_ani,average_containment_ani,max_containment_ani,n_weighted_found,total_weighted_hashes,species_name,acc
str,str,str,f64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,str,str
"""GCA_963606895 s__Methanocatell…","""98787da9033ca08f4a8e0e553cd3be…","""SRR11183330""",0.18467,465,21,1000,"""DNA""","""91703afddeec80707490192525488c…",0.001806,0.18467,10.012903,10.0,5.956098,0.922713,0.740505,0.831609,0.922713,4656,1066419,"""s__Methanocatella smithii""","""SRR11183330"""


In [6]:
# Do some nice reporting out of samples belong to a particular bioproject

class ReportOnBioProject:
    def __init__(self, *, manysearch_df, metadata_df, species_set, CUTOFF=9):
        all_acc = set(manysearch_df['acc'])
        assert len(all_acc) == 3216
        manysearch_df = manysearch_df.filter(pl.col('species_name').is_in(species_set))

        self.species_set = species_set
        self.manysearch_df = manysearch_df
        self.metadata_df = metadata_df

        by_acc = manysearch_df.group_by('acc').agg(
            n_core=pl.len(),
        )

        # how many in each bioproject are below cutoff?
        self.low_count_df = (
            by_acc
            .filter(pl.col('n_core') <= CUTOFF)
            .join(metadata_df, on='acc', how='inner')
            .group_by('bioproject').agg(n_low=pl.len())
        )
        # how many were searched in the 3216?
        self.manysearch_count_df = (
            metadata_df.filter(pl.col('acc').is_in(all_acc))
            .group_by('bioproject').agg(n_3216=pl.len())
        )
        # how many were in the SRA to begin with?
        self.sra_count_df = (
            metadata_df.group_by('bioproject').agg(n_sra=pl.len())
        )

    def report(self, *bioprojects, sort='n_low'):
        xx_df = (
            self.low_count_df
            .join(self.manysearch_count_df, on='bioproject', how='inner')
            .join(self.sra_count_df, on='bioproject', how='inner')
        )
        
        print(xx_df.filter(pl.col('bioproject').is_in(set(list(bioprojects)))).sort(sort))

reporter = ReportOnBioProject(manysearch_df=df, metadata_df=metadata_df, species_set=CORE_NAMES)
reporter.report('PRJNA668104')

shape: (1, 4)
┌─────────────┬───────┬────────┬───────┐
│ bioproject  ┆ n_low ┆ n_3216 ┆ n_sra │
│ ---         ┆ ---   ┆ ---    ┆ ---   │
│ str         ┆ u32   ┆ u32    ┆ u32   │
╞═════════════╪═══════╪════════╪═══════╡
│ PRJNA668104 ┆ 17    ┆ 21     ┆ 40    │
└─────────────┴───────┴────────┴───────┘


## Summarize by species to get core

In [7]:
n_acc = df['match_name'].n_unique()
by_species = df.group_by('query_name').agg(
    freq=pl.len() / n_acc
)

In [8]:
with pl.Config(tbl_rows=-1):
    print(by_species.sort('freq', descending=True).filter(pl.col('freq') >= 0.95))

shape: (17, 2)
┌─────────────────────────────────┬──────────┐
│ query_name                      ┆ freq     │
│ ---                             ┆ ---      │
│ str                             ┆ f64      │
╞═════════════════════════════════╪══════════╡
│ GCF_001256715 s__Escherichia c… ┆ 0.992537 │
│ GCA_945834365 s__Sodaliphilus … ┆ 0.990983 │
│ GCF_028308525 s__Lactobacillus… ┆ 0.990361 │
│ GCA_946007015 s__UBA2868 sp004… ┆ 0.979167 │
│ GCA_927798655 s__Cryptobactero… ┆ 0.978545 │
│ GCA_034171835 s__Mogibacterium… ┆ 0.977301 │
│ GCA_945872445 s__JAFBIX01 sp02… ┆ 0.974502 │
│ GCA_946408405 s__Fimisoma sp00… ┆ 0.973259 │
│ GCA_945501735 s__Prevotella sp… ┆ 0.968595 │
│ GCA_945876695 s__Floccifex por… ┆ 0.968284 │
│ GCA_004561115 s__Cryptobactero… ┆ 0.968284 │
│ GCA_945877245 s__Bariatricus s… ┆ 0.966418 │
│ GCA_022781365 s__Colivicinus s… ┆ 0.958333 │
│ GCA_945932635 s__Cryptobactero… ┆ 0.956468 │
│ GCA_945938975 s__Ornithospiroc… ┆ 0.955224 │
│ GCA_934726065 s__Cryptobactero… ┆ 0.952425 

In [9]:
CORE_NAMES = set(by_species.sort('freq', descending=True).filter(pl.col('freq') >= 0.95)['query_name'])
assert len(CORE_NAMES) == 17
print(CORE_NAMES)

{'GCA_022781365 s__Colivicinus sp002299675 singlehash', 'GCA_946007015 s__UBA2868 sp004552595 singlehash', 'GCA_934726065 s__Cryptobacteroides sp000432655 singlehash', 'GCA_945872445 s__JAFBIX01 sp021531895 singlehash', 'GCA_034171835 s__Mogibacterium_A kristiansenii singlehash', 'GCA_927798655 s__Cryptobacteroides sp900546925 singlehash', 'GCA_946408405 s__Fimisoma sp002320005 singlehash', 'GCF_028308525 s__Lactobacillus amylovorus singlehash', 'GCA_004561115 s__Cryptobacteroides sp034089285 singlehash', 'GCA_945877245 s__Bariatricus sp004560705 singlehash', 'GCA_945938975 s__Ornithospirochaeta sp022785155 singlehash', 'GCA_028727695 s__Prevotella sp000434975 singlehash', 'GCA_945876695 s__Floccifex porci singlehash', 'GCA_945932635 s__Cryptobacteroides sp000434935 singlehash', 'GCA_945501735 s__Prevotella sp002251295 singlehash', 'GCF_001256715 s__Escherichia coli singlehash', 'GCA_945834365 s__Sodaliphilus sp004557565 singlehash'}


## Ask questions about core species by metagenome

In [10]:
by_acc = (
    df
    .filter(pl.col('query_name').is_in(CORE_NAMES))
    .join(metadata_df, left_on='match_name', right_on='acc')
    .group_by('match_name').agg(
        bioproject=pl.col('bioproject').unique().get(0),
        n_core=pl.len(),
    )
).sort('n_core')
by_acc.head(1)

match_name,bioproject,n_core
str,str,u32
"""ERR3212024""","""PRJEB31650""",1


In [11]:
assert by_acc['match_name'].n_unique() == 3216 # we found at least _one_ matches in all 3216 metagenomes

In [12]:
by_acc

match_name,bioproject,n_core
str,str,u32
"""ERR3212024""","""PRJEB31650""",1
"""SRR12795785""","""PRJNA668104""",1
"""ERR3212036""","""PRJEB31650""",1
"""ERR3212035""","""PRJEB31650""",1
"""SRR12795790""","""PRJNA668104""",1
…,…,…
"""SRR16235699""","""PRJNA769425""",17
"""SRR8960102""","""PRJNA526405""",17
"""ERR1135309""","""PRJEB11755""",17


## Join in the metadata

In [13]:
by_acc2 = by_acc.join(metadata_df, left_on='match_name', right_on='acc', how='inner', coalesce=True)
by_acc2.head(1)

match_name,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR11125486""","""PRJNA526405""",17,"""Fe24/14208""",[],"""WGS""",302,"""PRJNA526405""","""SAMN11098298""","[""Metagenome or environmental""]","""UNIVERSITY OF TECHNOLOGY SYDNE…",2017-02-24,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX7761821""","""Australia""","""Oceania""","[""Australia: NSW""]",null,"""Illumina NovaSeq 6000""","""plate_3_D12""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1400,397,"""pig gut metagenome""","""ILLUMINA""",2020-02-20 00:00:00 UTC,"""SRS4498273""","""SRP188615""",null,null,null,null,"""[""2017-02-24""]""",null,"""[""Host- gut""]""",null,"""[""Faecal""]""",null,"""[""Faecal""]""",null,null,"""[""Porcine""]""",null,"""""Intestinal Tract""""",null,null,"""[""male""]""",null,null,null,null,"""""34.1273 S 150.7387 E""""",null,null,null,null,"""1400832738""","""416591685""","""""2020-02-20T21:25:00.000Z""""",null,"""""11098298"""""


## Do we see any strong patterns in which species is most often missing? No.

In [14]:
by_acc_16 = by_acc.filter(pl.col('n_core') == 16)
with pl.Config(tbl_rows=-1):
    print(df
 .filter(pl.col('query_name').is_in(CORE_NAMES))
 ['query_name'].value_counts().sort('count')
)
print(df['match_name'].n_unique())

shape: (17, 2)
┌─────────────────────────────────┬───────┐
│ query_name                      ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ GCA_028727695 s__Prevotella sp… ┆ 3057  │
│ GCA_934726065 s__Cryptobactero… ┆ 3063  │
│ GCA_945938975 s__Ornithospiroc… ┆ 3072  │
│ GCA_945932635 s__Cryptobactero… ┆ 3076  │
│ GCA_022781365 s__Colivicinus s… ┆ 3082  │
│ GCA_945877245 s__Bariatricus s… ┆ 3108  │
│ GCA_004561115 s__Cryptobactero… ┆ 3114  │
│ GCA_945876695 s__Floccifex por… ┆ 3114  │
│ GCA_945501735 s__Prevotella sp… ┆ 3115  │
│ GCA_946408405 s__Fimisoma sp00… ┆ 3130  │
│ GCA_945872445 s__JAFBIX01 sp02… ┆ 3134  │
│ GCA_034171835 s__Mogibacterium… ┆ 3143  │
│ GCA_927798655 s__Cryptobactero… ┆ 3147  │
│ GCA_946007015 s__UBA2868 sp004… ┆ 3149  │
│ GCF_028308525 s__Lactobacillus… ┆ 3185  │
│ GCA_945834365 s__Sodaliphilus … ┆ 3187  │
│ GCF_001256715 s__Escherichia c… ┆ 3192  │
└────────────────

## What's our distribution of numbers of core microbes present in metagenomes?

tl;dr relatively few metagenomes have few core microbes

In [15]:
with pl.Config(tbl_rows=-1):
    print(by_acc['n_core'].value_counts())

print(len(by_acc.filter(pl.col('n_core') < 17)))

shape: (17, 2)
┌────────┬───────┐
│ n_core ┆ count │
│ ---    ┆ ---   │
│ u32    ┆ u32   │
╞════════╪═══════╡
│ 1      ┆ 6     │
│ 2      ┆ 12    │
│ 3      ┆ 3     │
│ 4      ┆ 5     │
│ 5      ┆ 8     │
│ 6      ┆ 3     │
│ 7      ┆ 7     │
│ 8      ┆ 5     │
│ 9      ┆ 5     │
│ 10     ┆ 9     │
│ 11     ┆ 10    │
│ 12     ┆ 15    │
│ 13     ┆ 32    │
│ 14     ┆ 38    │
│ 15     ┆ 109   │
│ 16     ┆ 279   │
│ 17     ┆ 2670  │
└────────┴───────┘
546


## Let's pick 10 as an arbitrary cutoff to investigate

We could also use 8 (per branchwater combinations) or go up to ~11 or 12?

In [16]:
low_core_acc = by_acc.filter(pl.col('n_core') < 10)
len(low_core_acc)
low_core_acc.head(1)

match_name,bioproject,n_core
str,str,u32
"""ERR3212024""","""PRJEB31650""",1


In [17]:
low_core_acc = low_core_acc.rename({'match_name': 'acc'})

In [18]:
low_core_acc2 = low_core_acc.join(metadata_df, on='acc', how='left')
low_core_acc2

acc,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR3212024""","""PRJEB31650""",1,"""SAMEA5414670""","[""DTU2018_MG_1468_LPSX_NX1NS_P1pureMC_0h_a""]","""WGS""",295,"""PRJEB31650""","""SAMEA5414670""",[],"""CENTRE FOR GENOMIC EPIDEMIOLOG…",2016-04-11,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""ena"", ""gs"", … ""s3""]","[""ena"", ""gs.us-east1"", … ""s3.us-east-1""]","[""2019-03-13""]","[""2019-03-12""]","""ERX3239551""","""Denmark""","""Europe""",[],null,"""NextSeq 500""","""unspecified""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1825,796,"""pig gut metagenome""","""ILLUMINA""",2019-03-21 00:00:00 UTC,"""ERS3219971""","""ERP114226""",null,null,null,null,"""[""2016-04-11""]""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1825158267""","""835005957""","""""2021-11-04T09:07:00.000Z""""",null,"""""11156626"""""
"""SRR12795785""","""PRJNA668104""",1,"""pig_colon_403""",[],"""WGS""",302,"""PRJNA668104""","""SAMN16396804""","[""Metagenome or environmental""]","""UNIVERSITY OF COPENHAGEN""",2018-09-07,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX9264794""","""Denmark""","""Europe""","[""Denmark:Copenhagen""]",null,"""Illumina NovaSeq 6000""","""17119-05-04-243200101""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,8376,2605,"""pig gut metagenome""","""ILLUMINA""",2021-10-01 00:00:00 UTC,"""SRS7494371""","""SRP286761""",null,null,null,null,"""[""2018-09-07""]""",null,null,null,null,null,null,null,null,"""[""Crossbred piglets (Landrace …",null,null,null,null,null,null,null,null,null,"""""55.68 N 12.54 E""""",null,null,null,null,"""8376513600""","""2732017240""","""""2020-10-08T12:38:00.000Z""""",null,"""""16396804"""""
"""ERR3212036""","""PRJEB31650""",1,"""SAMEA5414682""","[""DTU2018_MG_1516_LPSX_NX2NS_P1pureMC_0h_c""]","""WGS""",297,"""PRJEB31650""","""SAMEA5414682""",[],"""CENTRE FOR GENOMIC EPIDEMIOLOG…",2016-04-11,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""ena"", ""gs"", … ""s3""]","[""ena"", ""gs.us-east1"", … ""s3.us-east-1""]","[""2019-03-13""]","[""2019-03-12""]","""ERX3239563""","""Denmark""","""Europe""",[],null,"""NextSeq 500""","""unspecified""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1544,672,"""pig gut metagenome""","""ILLUMINA""",2019-03-21 00:00:00 UTC,"""ERS3219983""","""ERP114226""",null,null,null,null,"""[""2016-04-11""]""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1544387611""","""704684139""","""""2021-11-04T10:00:00.000Z""""",null,"""""11156638"""""
"""ERR3212035""","""PRJEB31650""",1,"""SAMEA5414681""","[""DTU2018_MG_1515_LPSX_NX2NS_P1pureMC_0h_

In [19]:
# write out to a CSV for follow-on manual curation
blank_man_cur1 = low_core_acc2.select(
    ['acc', 'n_core', 'bioproject', 'organism']
).with_columns(
    broad_cat=pl.lit('pig'),  # all of these are _labelled_ as pig
    labelled_as_pig=pl.lit(1),
    is_actually_pig=pl.lit(None),
    is_pig_and_atypical=pl.lit(None),
    summary=pl.lit(None),
    description=pl.lit(None)
)
#blank_man_cur1.write_csv('../BLANK-curated-low-n_core.csv')

## Summarize by bioproject

In [44]:
reporter.report(*low_core_acc['bioproject'], sort='n_low')

shape: (9, 4)
┌─────────────┬───────┬────────┬───────┐
│ bioproject  ┆ n_low ┆ n_3216 ┆ n_sra │
│ ---         ┆ ---   ┆ ---    ┆ ---   │
│ str         ┆ u32   ┆ u32    ┆ u32   │
╞═════════════╪═══════╪════════╪═══════╡
│ PRJNA471402 ┆ 1     ┆ 36     ┆ 39    │
│ PRJNA629856 ┆ 1     ┆ 58     ┆ 179   │
│ PRJNA741980 ┆ 1     ┆ 14     ┆ 14    │
│ PRJNA408025 ┆ 3     ┆ 26     ┆ 47    │
│ PRJEB31650  ┆ 5     ┆ 245    ┆ 567   │
│ PRJNA807368 ┆ 7     ┆ 12     ┆ 12    │
│ PRJNA373834 ┆ 8     ┆ 21     ┆ 22    │
│ PRJNA526405 ┆ 14    ┆ 2103   ┆ 3598  │
│ PRJNA668104 ┆ 17    ┆ 21     ┆ 40    │
└─────────────┴───────┴────────┴───────┘


## 9 bioprojects are responsible for the 54 samples with 9 or fewer core species

In [25]:
descriptions = {}
descriptions['PRJNA668104'] = 'Fecal microbiota transplantation on necrotizing enterocolitis in preterm piglets'
descriptions['PRJNA526405'] = 'Study of pig microbiome in response to antibiotic'
descriptions['PRJNA373834'] = 'intestinal microbiome in feces from sows, five days after farrowing, their piglets during suckling and two weeks after weaning, 5-day old artificially reared and formula-fed siblings, and FP infected with Clostridium difficile.'
descriptions['PRJNA807368'] = 'This study aimed to investigate the microbial structure and function in the rectum of weaned piglets with berberine supplementation.'
descriptions['PRJEB31650'] = 'Based on 4 samples processed differently...'
descriptions['PRJNA408025'] = 'effects of LDA exposure'
descriptions['PRJNA741980'] = 'phage-encoded AMR'
descriptions['PRJNA629856'] = 'The effect of weaning age on the piglet gut microbiome was evaluated by collecting fecal swabs from piglets and sows.'
descriptions['PRJNA471402'] = 'antibiotic feed additive (oxytetracycline)'

## Explore PRJNA471402 - antibiotic feed additive (oxytetracycline)

no obvious paper

In [61]:
reporter.report('PRJNA471402')

shape: (1, 4)
┌─────────────┬───────┬────────┬───────┐
│ bioproject  ┆ n_low ┆ n_3216 ┆ n_sra │
│ ---         ┆ ---   ┆ ---    ┆ ---   │
│ str         ┆ u32   ┆ u32    ┆ u32   │
╞═════════════╪═══════╪════════╪═══════╡
│ PRJNA471402 ┆ 1     ┆ 36     ┆ 39    │
└─────────────┴───────┴────────┴───────┘


In [62]:
low_core_acc2.filter(pl.col('bioproject') == 'PRJNA471402')

acc,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR7182060""","""PRJNA471402""",5,"""NGS_B.T6.d0.43367328""",[],"""WGS""",290,"""PRJNA471402""","""SAMN09209542""","[""Metagenome or environmental""]","""BIOMIN RESEARCH CENTER""",2016-09-20,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX4098850""","""Austria""","""Europe""","[""Austria:Tulln""]",null,"""NextSeq 500""","""NGS_B_T6_d0""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1529,663,"""pig gut metagenome""","""ILLUMINA""",2018-05-18 00:00:00 UTC,"""SRS3315063""","""SRP148398""",null,null,null,null,"""[""2016-09-20""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa domesticus""]""",null,null,null,null,null,null,null,null,null,"""""48.3315 N 16.0607 E""""",null,null,null,null,"""1529205919""","""695415979""","""""2018-05-18T03:50:00.000Z""""",null,"""""471402"""""


## Explore PRJNA408025 - effects of LDA exposure

note, 47 in SRA, only 26 in 3216.

nothing stands out. no obvious paper, either??

In [59]:
reporter.report('PRJNA408025')

shape: (1, 4)
┌─────────────┬───────┬────────┬───────┐
│ bioproject  ┆ n_low ┆ n_3216 ┆ n_sra │
│ ---         ┆ ---   ┆ ---    ┆ ---   │
│ str         ┆ u32   ┆ u32    ┆ u32   │
╞═════════════╪═══════╪════════╪═══════╡
│ PRJNA408025 ┆ 3     ┆ 26     ┆ 47    │
└─────────────┴───────┴────────┴───────┘


In [60]:
low_core_acc2.filter(pl.col('bioproject') == 'PRJNA408025')

acc,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR11489765""","""PRJNA408025""",7,"""I-CON4""",[],"""WGS""",300,"""PRJNA408025""","""SAMN14543241""","[""Metagenome or environmental""]","""CHINESE ACADEMY OF AGRICULTURA…",2017-03-05,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX8065750""","""China""","""Asia""","[""China:Sichuan""]",null,"""Illumina HiSeq 3000""","""EGENE_LI-CON4""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,13889,5530,"""pig gut metagenome""","""ILLUMINA""",2020-05-01 00:00:00 UTC,"""SRS6434484""","""SRP118553""",null,null,null,null,"""[""2017-03-05""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""30.3 N 103.0 E""""",null,null,null,null,"""13889740500""","""5798708875""","""""2020-04-07T17:52:00.000Z""""",null,"""""14543241"""""
"""SRR11489775""","""PRJNA408025""",8,"""I-CON10""",[],"""WGS""",300,"""PRJNA408025""","""SAMN14543232""","[""Metagenome or environmental""]","""CHINESE ACADEMY OF AGRICULTURA…",2017-03-05,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX8065740""","""China""","""Asia""","[""China:Sichuan""]",null,"""Illumina HiSeq 3000""","""EGENE_LI-CON10""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,12839,5145,"""pig gut metagenome""","""ILLUMINA""",2020-05-01 00:00:00 UTC,"""SRS6434474""","""SRP118553""",null,null,null,null,"""[""2017-03-05""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""30.3 N 103.0 E""""",null,null,null,null,"""12839204700""","""5395169141""","""""2020-04-07T15:45:00.000Z""""",null,"""""14543232"""""
"""SRR11489751""","""PRJNA408025""",8,"""I-LDA9""",[],"""WGS""",300,"""PRJNA408025""","""SAMN14543254""","[""Metagenome or environmental""]","""CHINESE ACADEMY OF AGRICULTURA…",2017-03-05,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX8065764""","""China""","""Asia""","[""China:Sichuan""]",null,"""Illumina HiSeq 3000""","""EGENE_LI-LDA9""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,13844,5449,"""pig gut metagenome""","""ILLUMINA""",2020-05-01 00:00:00 UTC,"""SRS6434498""","""SRP118553""",null,null,null,null,"""[""2017-03-05""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""30.3 N 103.0 E""""",null,null,null,null,"""13844902500""","""5713795958""","""""2020-04-07T15:18:00.000Z""""",null,"""""14543254"""""


## Explore PRJNA741980 - phage-encoded AMR

https://europepmc.org/article/PMC/PMC9723715

nothing stands out - it's just pig 7, looks like?

In [49]:
reporter.report('PRJNA741980')

shape: (1, 4)
┌─────────────┬───────┬────────┬───────┐
│ bioproject  ┆ n_low ┆ n_3216 ┆ n_sra │
│ ---         ┆ ---   ┆ ---    ┆ ---   │
│ str         ┆ u32   ┆ u32    ┆ u32   │
╞═════════════╪═══════╪════════╪═══════╡
│ PRJNA741980 ┆ 1     ┆ 14     ┆ 14    │
└─────────────┴───────┴────────┴───────┘


In [51]:
low_core_acc2.filter(pl.col('bioproject') == 'PRJNA741980')

acc,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR15057923""","""PRJNA741980""",6,"""P7_R1_R2""",[],"""WGS""",250,"""PRJNA741980""","""SAMN20056604""","[""Metagenome or environmental""]","""INRAE""",null,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX11368236""","""France""","""Europe""","[""France: Brittany""]",null,"""Illumina HiSeq 3000""","""P7_R1_R2""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,4655,2489,"""pig gut metagenome""","""ILLUMINA""",2021-09-30 00:00:00 UTC,"""SRS9414443""","""SRP327064""",null,null,null,null,"""[""2015""]""",null,null,null,null,null,null,null,null,"""[""pig""]""",null,null,null,null,null,null,null,null,null,"""""missing""""",null,null,null,null,"""4655376250""","""2610321308""","""""2021-07-07T05:27:00.000Z""""",null,"""""20056604"""""


In [57]:
with pl.Config(tbl_rows=-1):
    print(metadata_df.filter(pl.col('bioproject') == 'PRJNA741980')['sample_name'].sort())

shape: (14,)
Series: 'sample_name' [str]
[
	"P11_R1_R2"
	"P12_R1_R2"
	"P13_R1_R2"
	"P15_R1_R2"
	"P16_R1_R2"
	"P17_R1_R2"
	"P2_R1_R2"
	"P3_R1_R2"
	"P4_R1_R2"
	"P5_R1_R2"
	"P6_R1_R2"
	"P7_R1_R2"
	"P8_R1_R2"
	"P9_R1_R2"
]


## Explore PRJNA629856 - weaning age, 1 sample

https://pmc.ncbi.nlm.nih.gov/articles/PMC8609972/

thoughts:
* not clear how to decode sample names.
* lots that didn't make it into the 3216...

In [48]:
reporter.report('PRJNA629856')

shape: (1, 4)
┌─────────────┬───────┬────────┬───────┐
│ bioproject  ┆ n_low ┆ n_3216 ┆ n_sra │
│ ---         ┆ ---   ┆ ---    ┆ ---   │
│ str         ┆ u32   ┆ u32    ┆ u32   │
╞═════════════╪═══════╪════════╪═══════╡
│ PRJNA629856 ┆ 1     ┆ 58     ┆ 179   │
└─────────────┴───────┴────────┴───────┘


In [47]:
low_core_acc2.filter(pl.col('bioproject') == 'PRJNA629856')

acc,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR14369226""","""PRJNA629856""",7,"""H1P910W28FD007""",[],"""WGS""",502,"""PRJNA629856""","""SAMN17899767""","[""MIMS.me"", ""MIGS/MIMS/MIMARKS.host-associated""]","""AGRICULTURE AND AGRI-FOOD CANA…",null,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX10721749""","""Canada""","""North America""","[""Canada:Lacombe""]",null,"""Illumina NovaSeq 6000""","""H1P910W28FD007""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,8400,2612,"""pig gut metagenome""","""ILLUMINA""",2021-05-06 00:00:00 UTC,"""SRS8812147""","""SRP262165""",null,null,null,null,"""[""2019""]""",null,null,"""[""not applicable""]""",null,"""[""not applicable""]""",null,"""[""not applicable""]""",null,"""[""Sus scrofa domesticus""]""","""[""7 d""]""","""""Rectum""""","""""Feces""""",null,"""[""female""]""","""""910""""",null,null,null,"""""52.4685 N 113.7307 W""""",null,null,null,null,"""8400714482""","""2738927544""","""""2021-04-30T13:57:00.000Z""""",null,"""""17899767"""""


## Explore PRJNA526405 (big Australian study)

TODO: add papers, link to metadata

[ENA link](https://www.ebi.ac.uk/ena/browser/view/PRJNA526405)

Study of pig microbiome in response to antibiotic. The big Aus one.

Maybe some relationship to Ja31/ prefix in sample_name?

In [26]:
(by_acc
     .join(metadata_df, left_on='match_name', right_on='acc', how='inner')
     .filter(pl.col('bioproject') == 'PRJNA526405')
     .sort('n_core')
     .select('n_core', 'sample_name')
     .filter(pl.col('sample_name').str.contains('Ja31/'))
     .filter(pl.col('n_core') < 10)
)

n_core,sample_name
u32,str
7,"""Ja31/14276"""
7,"""Ja31/29951"""
7,"""Ja31/14276"""
7,"""Ja31/29951"""
7,"""Ja31/29951"""
8,"""Ja31/14276"""
9,"""Ja31/29792"""
9,"""Ja31/29792"""


## Explore PRJEB31650 study (diff sample processing)

[ENA link](https://www.ebi.ac.uk/ena/browser/view/PRJEB31650)

Based on 4 samples processed differently (Pig feces 1 & 2 (P1 & P2) Sewage 1 & 2 (S1 & S2)), storage conditions (Time 0h, 16h, 64h Temperature (-80C, -20C, 5C, 22C)), sequencing platform (FTX=Freeze thawing, HX=Nextflex Hiseq, LTSX=Kappa Hiseq, NX1NS=Nextera Nextseq, NX2NS=Nextera Nextseq, NFNS=Nextflex Nextseq). Sample names containing HHKTMBBXX are resequenced samples. Sample names containing MC are spiked with a mock community. Sample names containing EC, ECPBS and pureMC, are negative DNA extraction controls, negative controls containing PBS and postive controls of the mock community used to spike with, respectively. The a, b and c are indicators of replicate number.

summary: all 9 are pureMC - positive controls of mock community used to spike. 

Composition: "synthetic mock community composed of eight microorganisms that included two eukaryotes (Propionibacterium freudenreichii, Bacteroides fragilis, Staphylococcus aureus, Fusobacterium nucleatum, Escherichia coli, Salmonella enterica serovar Typhimurium, Cryptosporidium parvum, and Saccharomyces cerevisiae), respectively."

Note: we do see a great deal of E. coli! But we also see low intersection amounts of a few others, what's up with that??

TODO: examine overlaps with other expected organisms. Note that there are only a few intersect hashes but HIGH DEPTH.

pubs:

https://journals.asm.org/doi/10.1128/spectrum.00090-22

https://journals.asm.org/doi/10.1128/spectrum.01387-21

In [27]:
(by_acc
     .join(metadata_df, left_on='match_name', right_on='acc', how='inner')
     .filter(pl.col('bioproject') == 'PRJEB31650')
     .sort('n_core')
     .select('n_core', 'sample_name_sam')
     .filter(pl.col('n_core') < 10)
)

n_core,sample_name_sam
u32,list[str]
1,"[""DTU2018_MG_1516_LPSX_NX2NS_P1pureMC_0h_c""]"
1,"[""DTU2018_MG_1468_LPSX_NX1NS_P1pureMC_0h_a""]"
1,"[""DTU2018_MG_1515_LPSX_NX2NS_P1pureMC_0h_a""]"
1,"[""DTU2018_MG_1469_LPSX_NX1NS_P1pureMC_0h_c""]"
4,"[""DTU2016_MG_474_HX_P1pureMC_0h_a""]"
4,"[""HHKTMBBXXDTU2016_MG_474_HX_P1pureMC_0h_a""]"
5,"[""DTU2018_MG_349_LPSX_KAHI_P1pureMC_0h_a""]"
5,"[""HHKTMBBXX-replacementsDTU2016_MG_475_HX_P1pureMC_0h_b""]"
5,"[""DTU2018_MG_350_LPSX_KAHI_P1pureMC_0h_c""]"


In [28]:
# which species _were_ found in these?!
this_df = df.join(
    # get those in low_core, from the relevant bioproject
    (low_core_acc
     .join(metadata_df, on='acc', how='inner')
     .filter(pl.col('bioproject') == 'PRJEB31650')
    ),
    left_on='match_name', right_on='acc',
    how='semi'
).filter(pl.col('query_name').is_in(CORE_NAMES))

with pl.Config(tbl_rows=-1):
    print(this_df.sort('intersect_hashes'))

shape: (27, 22)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ query_nam ┆ query_md5 ┆ match_nam ┆ containme ┆ … ┆ n_weighte ┆ total_wei ┆ species_n ┆ acc      │
│ e         ┆ ---       ┆ e         ┆ nt        ┆   ┆ d_found   ┆ ghted_has ┆ ame       ┆ ---      │
│ ---       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ hes       ┆ ---       ┆ str      │
│ str       ┆           ┆ str       ┆ f64       ┆   ┆ i64       ┆ ---       ┆ str       ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆ i64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ GCA_02872 ┆ 5f3ec5317 ┆ ERR321174 ┆ 0.00288   ┆ … ┆ 23        ┆ 915385    ┆ s__Prevot ┆ ERR32117 │
│ 7695 s__P ┆ 3d835a3cb ┆ 7         ┆           ┆   ┆           ┆           ┆ ella sp00 ┆ 47       │
│ revotella ┆ 129557502 ┆           ┆           ┆   ┆           ┆          

## Explore PRJNA373834 study (Cdiff/formula)

"intestinal microbiome in feces from sows, five days after farrowing, their piglets during suckling and two weeks after weaning, 5-day old artificially reared and formula-fed siblings, and FP infected with Clostridium difficile"

no publication?

summary: all of the formula fed piglets < 10 core. also two of the suckling piglets.

In [29]:
biosample_info = """\
SAMN06315441	Weaned Piglet 3
SAMN06315439	Weaned Piglet 1
SAMN06315460	Sow 4
SAMN06315449	Suckling Piglet 3
SAMN06315450	Suckling Piglet 4
SAMN06315453	Formula fed 3
SAMN06315440	Weaned Piglet 2
SAMN06315455	Formula fed cdiff 2
SAMN06315454	Formula fed cdiff 1
SAMN06315443	Weaned Piglet 5
SAMN06315452	Formula fed 2
SAMN06315446	Weaned Piglet 8
SAMN06315447	Suckling Piglet 1
SAMN06315445	Weaned Piglet 7
SAMN06315456	Formula fed cdiff 3
SAMN06315451	Formula fed 1
SAMN06315458	Sow 2
SAMN06315459	Sow 3
SAMN06315448	Suckling Piglet 2
SAMN06315444	Weaned Piglet 6
SAMN06315442	Weaned Piglet 4
SAMN06315457	Sow 1
"""

xx = []
for line in biosample_info.splitlines():
    biosample, descr = line.strip().split('	')
    is_formula=0
    if 'formula' in descr.lower():
        is_formula=1
    is_cdiff=0
    if 'cdiff' in descr.lower():
        is_cdiff= 1
    xx.append(dict(biosample=biosample, descr=descr, is_formula=is_formula, is_cdiff=is_cdiff))

biosample_df = pl.DataFrame(xx)
with pl.Config(tbl_rows=-1):
    print(biosample_df)

shape: (22, 4)
┌──────────────┬─────────────────────┬────────────┬──────────┐
│ biosample    ┆ descr               ┆ is_formula ┆ is_cdiff │
│ ---          ┆ ---                 ┆ ---        ┆ ---      │
│ str          ┆ str                 ┆ i64        ┆ i64      │
╞══════════════╪═════════════════════╪════════════╪══════════╡
│ SAMN06315441 ┆ Weaned Piglet 3     ┆ 0          ┆ 0        │
│ SAMN06315439 ┆ Weaned Piglet 1     ┆ 0          ┆ 0        │
│ SAMN06315460 ┆ Sow 4               ┆ 0          ┆ 0        │
│ SAMN06315449 ┆ Suckling Piglet 3   ┆ 0          ┆ 0        │
│ SAMN06315450 ┆ Suckling Piglet 4   ┆ 0          ┆ 0        │
│ SAMN06315453 ┆ Formula fed 3       ┆ 1          ┆ 0        │
│ SAMN06315440 ┆ Weaned Piglet 2     ┆ 0          ┆ 0        │
│ SAMN06315455 ┆ Formula fed cdiff 2 ┆ 1          ┆ 1        │
│ SAMN06315454 ┆ Formula fed cdiff 1 ┆ 1          ┆ 1        │
│ SAMN06315443 ┆ Weaned Piglet 5     ┆ 0          ┆ 0        │
│ SAMN06315452 ┆ Formula fed 2       ┆ 1

In [30]:
with pl.Config(tbl_rows=-1):
    print(by_acc
     .join(metadata_df, left_on='match_name', right_on='acc', how='inner')
     .filter(pl.col('bioproject') == 'PRJNA373834')
     .join(biosample_df, on='biosample', how='inner')
     .select(['n_core', 'biosample', 'match_name', 'descr', 'is_cdiff', 'is_formula'])
     .sort('is_cdiff', 'is_formula')
    )

shape: (21, 6)
┌────────┬──────────────┬────────────┬─────────────────────┬──────────┬────────────┐
│ n_core ┆ biosample    ┆ match_name ┆ descr               ┆ is_cdiff ┆ is_formula │
│ ---    ┆ ---          ┆ ---        ┆ ---                 ┆ ---      ┆ ---        │
│ u32    ┆ str          ┆ str        ┆ str                 ┆ i64      ┆ i64        │
╞════════╪══════════════╪════════════╪═════════════════════╪══════════╪════════════╡
│ 17     ┆ SAMN06315459 ┆ SRR5240726 ┆ Sow 3               ┆ 0        ┆ 0          │
│ 17     ┆ SAMN06315439 ┆ SRR5240751 ┆ Weaned Piglet 1     ┆ 0        ┆ 0          │
│ 17     ┆ SAMN06315460 ┆ SRR5240725 ┆ Sow 4               ┆ 0        ┆ 0          │
│ 8      ┆ SAMN06315449 ┆ SRR5240737 ┆ Suckling Piglet 3   ┆ 0        ┆ 0          │
│ 16     ┆ SAMN06315450 ┆ SRR5240736 ┆ Suckling Piglet 4   ┆ 0        ┆ 0          │
│ 17     ┆ SAMN06315441 ┆ SRR5240749 ┆ Weaned Piglet 3     ┆ 0        ┆ 0          │
│ 17     ┆ SAMN06315440 ┆ SRR5240750 ┆ Weaned Pigl

In [31]:
(low_core_acc2.filter(pl.col('bioproject') == 'PRJNA373834')
 .join(biosample_df, on='biosample', how='inner')
 .select(['n_core', 'biosample', 'descr', 'is_cdiff', 'is_formula'])
)

n_core,biosample,descr,is_cdiff,is_formula
u32,str,str,i64,i64
8,"""SAMN06315449""","""Suckling Piglet 3""",0,0
6,"""SAMN06315453""","""Formula fed 3""",0,1
5,"""SAMN06315455""","""Formula fed cdiff 2""",1,1
6,"""SAMN06315454""","""Formula fed cdiff 1""",1,1
5,"""SAMN06315452""","""Formula fed 2""",0,1
5,"""SAMN06315456""","""Formula fed cdiff 3""",1,1
4,"""SAMN06315451""","""Formula fed 1""",0,1
9,"""SAMN06315448""","""Suckling Piglet 2""",0,0


In [32]:
low_core_acc2.filter(pl.col('bioproject') == 'PRJNA373834')

acc,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR5240735""","""PRJNA373834""",4,"""2016_CRC_FP1""",[],"""WGS""",302,"""PRJNA373834""","""SAMN06315451""","[""Metagenome or environmental""]","""FREIE UNIVERSITY OF BERLIN""",2016-08-01,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX2547607""","""Germany""","""Europe""","[""Germany: Berlin""]",null,"""NextSeq 500""","""L1_S20""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,4641,1838,"""pig gut metagenome""","""ILLUMINA""",2017-05-01 00:00:00 UTC,"""SRS1966284""","""SRP099123""",null,null,null,null,"""[""2016-08""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""52.458569 N 13.288657 E""""",null,null,null,null,"""4641704364""","""1927698411""","""""2017-02-09T10:39:00.000Z""""",null,"""""2016_CRC_FP1"""""
"""SRR5240734""","""PRJNA373834""",5,"""2016_CRC_FP2""",[],"""WGS""",302,"""PRJNA373834""","""SAMN06315452""","[""Metagenome or environmental""]","""FREIE UNIVERSITY OF BERLIN""",2016-08-01,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX2547606""","""Germany""","""Europe""","[""Germany: Berlin""]",null,"""NextSeq 500""","""L1_S21""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,4561,1785,"""pig gut metagenome""","""ILLUMINA""",2017-05-01 00:00:00 UTC,"""SRS1966283""","""SRP099123""",null,null,null,null,"""[""2016-08""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""52.458569 N 13.288657 E""""",null,null,null,null,"""4561089692""","""1872387541""","""""2017-02-09T10:45:00.000Z""""",null,"""""2016_CRC_FP2"""""
"""SRR5240729""","""PRJNA373834""",5,"""2016_CRC_FP-CD3""",[],"""WGS""",302,"""PRJNA373834""","""SAMN06315456""","[""Metagenome or environmental""]","""FREIE UNIVERSITY OF BERLIN""",2016-08-01,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX2547601""","""Germany""","""Europe""","[""Germany: Berlin""]",null,"""NextSeq 500""","""L1_S19""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,4273,1692,"""pig gut metagenome""","""ILLUMINA""",2017-05-01 00:00:00 UTC,"""SRS1966278""","""SRP099123""",null,null,null,null,"""[""2016-08""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""52.458569 N 13.288657 E""""",null,null,null,null,"""4273492374""","""1775136009""","""""2017-02-09T10:37:00.000Z""""",null,"""""2016_CRC_FP-CD3"""""
"""SRR5240730""","""PRJNA373834""",5,"""2016_CRC_FP-CD2""",[],"""WGS""",302

## Explore PRJNA668104 (necrotizing enterocolitis study)

[ENA link](https://www.ebi.ac.uk/ena/browser/view/PRJNA668104)

publication: https://www.nature.com/articles/s41522-022-00310-2

github: https://github.com/yanhui09/FMT-donor/tree/master

metadata link: https://github.com/yanhui09/FMT-donor/blob/master/data/Metadata.tsv

Weird: it looks like we didn't include all 40 in the 3216 metagenomes being searched? only 19?
e.g. SRR12795742 and SRR12795718 are not in the 3216 list. Why not?? MAYBE HOST CONTAM.

In [33]:
biosample_text = StringIO('''accession	description
SAMN16396816	FMT1_pig_1
SAMN16396823	FMT1_pig_8
SAMN16396815	DONOR2
SAMN16396805	CON_pig_4
SAMN16396829	FMT2_pig_1
SAMN16396834	FMT2_pig_6
SAMN16396802	CON_pig_1
SAMN16396814	DONOR1
SAMN16396820	FMT1_pig_5
SAMN16396837	FMT2_pig_9
SAMN16396808	CON_pig_7
SAMN16396839	FMT2_pig_11
SAMN16396840	FMT2_pig_12
SAMN16396813	CON_pig_12
SAMN16396835	FMT2_pig_7
SAMN16396826	FMT1_pig_11
SAMN16396819	FMT1_pig_4
SAMN16396818	FMT1_pig_3
SAMN16396828	FMT1_pig_13
SAMN16396807	CON_pig_6
SAMN16396822	FMT1_pig_7
SAMN16396832	FMT2_pig_4
SAMN16396806	CON_pig_5
SAMN16396824	FMT1_pig_9
SAMN16396817	FMT1_pig_2
SAMN16396831	FMT2_pig_3
SAMN16396836	FMT2_pig_8
SAMN16396811	CON_pig_10
SAMN16396821	FMT1_pig_6
SAMN16396809	CON_pig_8
SAMN16396810	CON_pig_9
SAMN16396827	FMT1_pig_12
SAMN16396830	FMT2_pig_2
SAMN16396803	CON_pig_2
SAMN16396838	FMT2_pig_10
SAMN16396804	CON_pig_3
SAMN16396812	CON_pig_11
SAMN16396825	FMT1_pig_10
SAMN16396841	FMT2_pig_13
SAMN16396833	FMT2_pig_5
''')

biosample_df = pl.read_csv(biosample_text, separator='\t').rename({ 'accession': 'biosample' })
biosample_df

biosample,description
str,str
"""SAMN16396816""","""FMT1_pig_1"""
"""SAMN16396823""","""FMT1_pig_8"""
"""SAMN16396815""","""DONOR2"""
"""SAMN16396805""","""CON_pig_4"""
"""SAMN16396829""","""FMT2_pig_1"""
…,…
"""SAMN16396804""","""CON_pig_3"""
"""SAMN16396812""","""CON_pig_11"""
"""SAMN16396825""","""FMT1_pig_10"""


In [34]:
with pl.Config(tbl_rows=-1):
    print(low_core_acc2
 .filter(pl.col('bioproject') == 'PRJNA668104')
 .join(biosample_df, on='biosample', how='inner')
 .select(['acc', 'n_core','description'])
 .sort('description')
 )

shape: (19, 3)
┌─────────────┬────────┬─────────────┐
│ acc         ┆ n_core ┆ description │
│ ---         ┆ ---    ┆ ---         │
│ str         ┆ u32    ┆ str         │
╞═════════════╪════════╪═════════════╡
│ SRR12795785 ┆ 1      ┆ CON_pig_3   │
│ SRR12795774 ┆ 2      ┆ CON_pig_4   │
│ SRR12795740 ┆ 2      ┆ CON_pig_5   │
│ SRR12795729 ┆ 2      ┆ CON_pig_6   │
│ SRR12795780 ┆ 2      ┆ FMT1_pig_11 │
│ SRR12795790 ┆ 1      ┆ FMT1_pig_2  │
│ SRR12795789 ┆ 2      ┆ FMT1_pig_3  │
│ SRR12795784 ┆ 2      ┆ FMT1_pig_7  │
│ SRR12795783 ┆ 2      ┆ FMT1_pig_8  │
│ SRR12795782 ┆ 2      ┆ FMT1_pig_9  │
│ SRR12795744 ┆ 2      ┆ FMT2_pig_10 │
│ SRR12795741 ┆ 2      ┆ FMT2_pig_13 │
│ SRR12795776 ┆ 4      ┆ FMT2_pig_2  │
│ SRR12795775 ┆ 2      ┆ FMT2_pig_3  │
│ SRR12795773 ┆ 3      ┆ FMT2_pig_4  │
│ SRR12795771 ┆ 4      ┆ FMT2_pig_6  │
│ SRR12795770 ┆ 3      ┆ FMT2_pig_7  │
│ SRR12795746 ┆ 3      ┆ FMT2_pig_8  │
│ SRR12795745 ┆ 2      ┆ FMT2_pig_9  │
└─────────────┴────────┴─────────────┘


In [35]:
with pl.Config(tbl_rows=-1):
    print(biosample_df.sort('description'))

shape: (40, 2)
┌──────────────┬─────────────┐
│ biosample    ┆ description │
│ ---          ┆ ---         │
│ str          ┆ str         │
╞══════════════╪═════════════╡
│ SAMN16396802 ┆ CON_pig_1   │
│ SAMN16396811 ┆ CON_pig_10  │
│ SAMN16396812 ┆ CON_pig_11  │
│ SAMN16396813 ┆ CON_pig_12  │
│ SAMN16396803 ┆ CON_pig_2   │
│ SAMN16396804 ┆ CON_pig_3   │
│ SAMN16396805 ┆ CON_pig_4   │
│ SAMN16396806 ┆ CON_pig_5   │
│ SAMN16396807 ┆ CON_pig_6   │
│ SAMN16396808 ┆ CON_pig_7   │
│ SAMN16396809 ┆ CON_pig_8   │
│ SAMN16396810 ┆ CON_pig_9   │
│ SAMN16396814 ┆ DONOR1      │
│ SAMN16396815 ┆ DONOR2      │
│ SAMN16396816 ┆ FMT1_pig_1  │
│ SAMN16396825 ┆ FMT1_pig_10 │
│ SAMN16396826 ┆ FMT1_pig_11 │
│ SAMN16396827 ┆ FMT1_pig_12 │
│ SAMN16396828 ┆ FMT1_pig_13 │
│ SAMN16396817 ┆ FMT1_pig_2  │
│ SAMN16396818 ┆ FMT1_pig_3  │
│ SAMN16396819 ┆ FMT1_pig_4  │
│ SAMN16396820 ┆ FMT1_pig_5  │
│ SAMN16396821 ┆ FMT1_pig_6  │
│ SAMN16396822 ┆ FMT1_pig_7  │
│ SAMN16396823 ┆ FMT1_pig_8  │
│ SAMN16396824 ┆ FMT1_pi

In [36]:
# from https://github.com/yanhui09/FMT-donor/blob/master/data/Metadata.tsv
github_meta_text = StringIO("""""	"CustomerID"	"FMT_ID"	"Litter"	"Sex"	"Group_old"	"GroupID"	"Group"	"NEC"	"NEC_max"	"NEC_severity"
"17119-05-01-243200068"	"0"	"400"	"1"	"m"	"CON4"	"CON4_1"	"CON"	0	3	2.33333333333333
"17119-05-02-243200066"	"1"	"401"	"1"	"m"	"CON4"	"CON4_2"	"CON"	1	6	6
"17119-05-03-243200095"	"2"	"402"	"1"	"m"	"NEW4"	"NEW4_1"	"FMT1"	1	5	3.33333333333333
"17119-05-04-243200101"	"3"	"403"	"1"	"f"	"CON4"	"CON4_3"	"CON"	0	3	1.66666666666667
"17119-05-05-243200087"	"4"	"404"	"1"	"f"	"OLD4"	"OLD4_1"	"FMT2"	0	1	1
"17119-05-06-243200069"	"5"	"405"	"1"	"f"	"NEW4"	"NEW4_2"	"FMT1"	0	1	1
"17119-05-07-243200099"	"6"	"406"	"1"	"f"	"CON4"	"CON4_4"	"CON"	0	2	1.66666666666667
"17119-05-08-243200071"	"7"	"407"	"1"	"m"	"OLD4"	"OLD4_2"	"FMT2"	0	2	1.33333333333333
"17119-05-09-243200070"	"8"	"408"	"1"	"f"	"NEW4"	"NEW4_3"	"FMT1"	0	1	1
"17119-05-10-243200096"	"9"	"409"	"1"	"f"	"CON4"	"CON4_5"	"CON"	0	2	1.33333333333333
"17119-05-11-243200091"	"10"	"410"	"1"	"f"	"OLD4"	"OLD4_3"	"FMT2"	0	2	1.66666666666667
"17119-05-12-243200081"	"11"	"411"	"1"	"f"	"NEW4"	"NEW4_4"	"FMT1"	0	2	1.33333333333333
"17119-05-13-243200076"	"12"	"412"	"1"	"m"	"CON4"	"CON4_6"	"CON"	1	5	4
"17119-05-14-243200089"	"13"	"413"	"1"	"f"	"OLD4"	"OLD4_4"	"FMT2"	0	1	1
"17119-05-15-243200093"	"14"	"414"	"1"	"m"	"NEW4"	"NEW4_5"	"FMT1"	0	1	1
"17119-05-16-243200072"	"15"	"415"	"1"	"m"	"CON4"	"CON4_7"	"CON"	1	6	4.33333333333333
"17119-05-17-243200086"	"16"	"416"	"1"	"m"	"OLD4"	"OLD4_5"	"FMT2"	0	1	1
"17119-05-18-243200100"	"17"	"417"	"1"	"m"	"NEW4"	"NEW4_6"	"FMT1"	1	5	3.33333333333333
"17119-05-19-243200062"	"18"	"418"	"1"	"m"	"OLD4"	"OLD4_6"	"FMT2"	0	2	1.66666666666667
"17119-05-20-243200063"	"19"	"419"	"1"	"m"	"NEW4"	"NEW4_7"	"FMT1"	1	5	2.33333333333333
"17119-05-21-243200075"	"20"	"420"	"1"	"m"	"OLD4"	"OLD4_7"	"FMT2"	0	2	1.33333333333333
"17119-05-22-243200094"	"21"	"421"	"1"	"m"	"NEW4"	"NEW4_8"	"FMT1"	0	3	1.66666666666667
"17119-05-23-243200073"	"22"	"422"	"2"	"f"	"OLD4"	"OLD4_8"	"FMT2"	0	2	1.33333333333333
"17119-05-24-243200088"	"23"	"423"	"2"	"f"	"CON4"	"CON4_8"	"CON"	1	5	2.33333333333333
"17119-05-25-243200085"	"24"	"424"	"2"	"f"	"NEW4"	"NEW4_9"	"FMT1"	0	1	1
"17119-05-26-243200098"	"25"	"425"	"2"	"f"	"OLD4"	"OLD4_9"	"FMT2"	0	3	1.66666666666667
"17119-05-27-243200083"	"26"	"426"	"2"	"f"	"CON4"	"CON4_9"	"CON"	0	1	1
"17119-05-28-243200078"	"27"	"427"	"2"	"f"	"NEW4"	"NEW4_10"	"FMT1"	0	1	1
"17119D-05-29-243200065"	"28"	"428"	"2"	"f"	"OLD4"	"OLD4_13"	"FMT2"	0	1	1
"17119-05-30-243200082"	"29"	"429"	"2"	"f"	"CON4"	"CON4_10"	"CON"	0	1	1
"17119-05-31-243200080"	"30"	"430"	"2"	"f"	"NEW4"	"NEW4_11"	"FMT1"	0	2	1.33333333333333
"17119-05-32-243200092"	"31"	"431"	"2"	"f"	"OLD4"	"OLD4_10"	"FMT2"	0	1	1
"17119-05-33-243200064"	"32"	"432"	"2"	"f"	"CON4"	"CON4_11"	"CON"	1	6	4
"17119-05-34-243200097"	"33"	"433"	"2"	"f"	"NEW4"	"NEW4_12"	"FMT1"	0	3	1.66666666666667
"17119-05-35-243200067"	"34"	"434"	"2"	"m"	"OLD4"	"OLD4_11"	"FMT2"	0	2	1.33333333333333
"17119-05-36-243200079"	"36"	"436"	"2"	"m"	"NEW4"	"NEW4_13"	"FMT1"	0	2	1.33333333333333
"17119-05-37-243200074"	"37"	"437"	"2"	"m"	"OLD4"	"OLD4_12"	"FMT2"	0	1	1
"17119-05-38-243200077"	"38"	"438"	"2"	"m"	"CON4"	"CON4_12"	"CON"	0	1	1
"17119-05-40-243200090"	"new"	"new"	"DONOR1"	"new"	"NEW Donor"	"NEW_Donor"	"DONOR1"	0	0	0
"17119-05-39-243200084"	"old"	"old"	"DONOR2"	"old"	"OLD Donor"	"OLD_Donor"	"DONOR2"	0	0	0
""")
github_meta_df = pl.read_csv(github_meta_text, separator='\t').rename({'': 'timestamp'})
github_meta_df

timestamp,CustomerID,FMT_ID,Litter,Sex,Group_old,GroupID,Group,NEC,NEC_max,NEC_severity
str,str,str,str,str,str,str,str,i64,i64,f64
"""17119-05-01-243200068""","""0""","""400""","""1""","""m""","""CON4""","""CON4_1""","""CON""",0,3,2.333333
"""17119-05-02-243200066""","""1""","""401""","""1""","""m""","""CON4""","""CON4_2""","""CON""",1,6,6.0
"""17119-05-03-243200095""","""2""","""402""","""1""","""m""","""NEW4""","""NEW4_1""","""FMT1""",1,5,3.333333
"""17119-05-04-243200101""","""3""","""403""","""1""","""f""","""CON4""","""CON4_3""","""CON""",0,3,1.666667
"""17119-05-05-243200087""","""4""","""404""","""1""","""f""","""OLD4""","""OLD4_1""","""FMT2""",0,1,1.0
…,…,…,…,…,…,…,…,…,…,…
"""17119-05-36-243200079""","""36""","""436""","""2""","""m""","""NEW4""","""NEW4_13""","""FMT1""",0,2,1.333333
"""17119-05-37-243200074""","""37""","""437""","""2""","""m""","""OLD4""","""OLD4_12""","""FMT2""",0,1,1.0
"""17119-05-38-243200077""","""38""","""438""","""2""","""m""","""CON4""","""CON4_12""","""CON""",0,1,1.0


In [37]:
xx_df = (low_core_acc2
 .filter(pl.col('bioproject') == 'PRJNA668104')
 .join(biosample_df, on='biosample', how='inner')
 .with_columns(
     descr2=pl.col('description').str.split('_').list.get(0),
 )
 .select(['acc', 'bioproject', 'n_core', 'sample_name', 'biosample', 'description', 'descr2'])
 )

In [38]:
by_acc2

match_name,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR11125486""","""PRJNA526405""",17,"""Fe24/14208""",[],"""WGS""",302,"""PRJNA526405""","""SAMN11098298""","[""Metagenome or environmental""]","""UNIVERSITY OF TECHNOLOGY SYDNE…",2017-02-24,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX7761821""","""Australia""","""Oceania""","[""Australia: NSW""]",null,"""Illumina NovaSeq 6000""","""plate_3_D12""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1400,397,"""pig gut metagenome""","""ILLUMINA""",2020-02-20 00:00:00 UTC,"""SRS4498273""","""SRP188615""",null,null,null,null,"""[""2017-02-24""]""",null,"""[""Host- gut""]""",null,"""[""Faecal""]""",null,"""[""Faecal""]""",null,null,"""[""Porcine""]""",null,"""""Intestinal Tract""""",null,null,"""[""male""]""",null,null,null,null,"""""34.1273 S 150.7387 E""""",null,null,null,null,"""1400832738""","""416591685""","""""2020-02-20T21:25:00.000Z""""",null,"""""11098298"""""
"""ERR1135387""","""PRJEB11755""",17,"""SAMEA3663214""","[""BHZ-7B""]","""WGS""",195,"""PRJEB11755""","""SAMEA3663214""",[],"""BEIJING GENOME INSTITUTE""",null,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""ena"", ""gs"", … ""s3""]","[""ena"", ""gs.us-east1"", … ""s3.us-east-1""]","[""2016-01-11""]","[""2018-11-16""]","""ERX1214566""","""China""","""Asia""",[],null,"""Illumina HiSeq 2000""","""130421""","""PAIRED""","""other""","""METAGENOMIC""",null,5827,4393,"""pig gut metagenome""","""ILLUMINA""",2016-01-12 00:00:00 UTC,"""ERS970363""","""ERP013165""",null,null,null,null,"""[""2013""]""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""[""metagenome""]""",null,null,"""[""A Catalogue of the pig Gut M…",null,null,null,"""5827806425""","""4607328246""","""""2016-01-13T01:23:00.000Z""""",null,"""""308698"""""
"""SRR11126080""","""PRJNA526405""",17,"""Fe24/14208""",[],"""WGS""",302,"""PRJNA526405""","""SAMN11098298""","[""Metagenome or environmental""]","""UNIVERSITY OF TECHNOLOGY SYDNE…",2017-02-24,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX7762511""","""Australia""","""Oceania""","[""Australia: NSW""]",null,"""Illumina NovaSeq 6000""","""plate_3_D12""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1389,404,"""pig gut metagenome""","""ILLUMINA""",2020-02-20 00:00:00 UTC,"""SRS4498273""","""SRP188615""",null,null,null,null,"""[""2017-02-24""]""",null,"""[""Host- gut""]""",null,"""[""Faecal""]""",null,"""[""Faecal""]""",null,null,"""[""Porcine""]""",null,"""""Intestinal Tract""""",null,null,"""[""male""]""",null,null,null,null,"""""34.1273 S 150.7387 E""""",null,null,null,null,"""1389296640"

In [39]:
# not all samples are included!? k anyway.
metadata_df.filter(pl.col('bioproject') == 'PRJNA668104').select(['acc']).sort('acc')

with open('PRJNA668104.acc.list', 'wt') as fp:
    for acc in metadata_df.filter(pl.col('bioproject') == 'PRJNA668104')['acc']:
        print(f'/group/ctbrowngrp5/wort/wort-sra/sigs/{acc}.sig', file=fp)

In [40]:
xx_df = (by_acc2
 .filter(pl.col('bioproject') == 'PRJNA668104')
 .join(biosample_df, on='biosample', how='inner')
 .with_columns(
     descr2=pl.col('description').str.split('_').list.get(0),
     sample_name2=pl.col('sample_name').str.split('_').list.get(-1),
 )
 .with_columns(# replace 'donor1' with 'new'
     sample_name2=pl.when(pl.col('sample_name2')=='donor1').then(pl.lit('new')).otherwise(pl.col('sample_name2'))
 )
 .with_columns(# replace 'donor2' with 'old'
     sample_name2=pl.when(pl.col('sample_name2')=='donor2').then(pl.lit('old')).otherwise(pl.col('sample_name2'))
 )
 .join(github_meta_df, left_on='sample_name2', right_on='FMT_ID')
# .select(['match_name', 'bioproject', 'n_core', 'sample_name', 'sample_name2', 'biosample', 'description', 'descr2'])
 .sort('descr2')
 )
with pl.Config(tbl_rows=-1):
    print(xx_df)

shape: (21, 82)
┌─────────────┬─────────────┬────────┬───────────────┬───┬────────┬─────┬─────────┬──────────────┐
│ match_name  ┆ bioproject  ┆ n_core ┆ sample_name   ┆ … ┆ Group  ┆ NEC ┆ NEC_max ┆ NEC_severity │
│ ---         ┆ ---         ┆ ---    ┆ ---           ┆   ┆ ---    ┆ --- ┆ ---     ┆ ---          │
│ str         ┆ str         ┆ u32    ┆ str           ┆   ┆ str    ┆ i64 ┆ i64     ┆ f64          │
╞═════════════╪═════════════╪════════╪═══════════════╪═══╪════════╪═════╪═════════╪══════════════╡
│ SRR12795785 ┆ PRJNA668104 ┆ 1      ┆ pig_colon_403 ┆ … ┆ CON    ┆ 0   ┆ 3       ┆ 1.666667     │
│ SRR12795774 ┆ PRJNA668104 ┆ 2      ┆ pig_colon_406 ┆ … ┆ CON    ┆ 0   ┆ 2       ┆ 1.666667     │
│ SRR12795740 ┆ PRJNA668104 ┆ 2      ┆ pig_colon_409 ┆ … ┆ CON    ┆ 0   ┆ 2       ┆ 1.333333     │
│ SRR12795729 ┆ PRJNA668104 ┆ 2      ┆ pig_colon_412 ┆ … ┆ CON    ┆ 1   ┆ 5       ┆ 4.0          │
│ SRR12795793 ┆ PRJNA668104 ┆ 17     ┆ donor1        ┆ … ┆ DONOR1 ┆ 0   ┆ 0       ┆ 0.0      

## Explore PRJNA807368 (berberine study)

[ENA link](https://www.ebi.ac.uk/ena/browser/view/PRJNA807368)

paper: https://www.frontiersin.org/journals/microbiology/articles/10.3389/fmicb.2022.862882/full

Summary of the below: of the 12 samples in the study, 4/6 of the Berberine suppl samples had < 10 core microbes, and none of the non-supplemented. All 6 berberine suppl samples had 10 core microbes or less, and only one control.

In [41]:
low_core_acc2.filter(pl.col('bioproject') == 'PRJNA807368')

acc,bioproject,n_core,sample_name,sample_name_sam,assay_type,avgspotlen,bioproject_right,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search
str,str,u32,str,list[str],str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR18048904""","""PRJNA807368""",5,"""Berberine1""",[],"""WGS""",300,"""PRJNA807368""","""SAMN25980738""","[""Metagenome or environmental""]","""ANHUI SCIENCE AND TECHNOLOGY U…",2021-12-10,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX14201178""","""China""","""Asia""","[""China: Bengbu""]",null,"""Illumina NovaSeq 6000""","""Berberine1""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,10226,3313,"""pig gut metagenome""","""ILLUMINA""",2022-02-18 00:00:00 UTC,"""SRS12021757""","""SRP360093""",null,null,null,null,"""[""2021-12-10""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa domesticus""]""",null,null,null,null,null,null,null,null,null,"""""Not collected""""",null,null,null,null,"""10226653500""","""3474421145""","""""2022-02-18T08:23:00.000Z""""",null,"""""25980738"""""
"""SRR18048903""","""PRJNA807368""",8,"""Berberine2""",[],"""WGS""",300,"""PRJNA807368""","""SAMN25980739""","[""Metagenome or environmental""]","""ANHUI SCIENCE AND TECHNOLOGY U…",2021-12-10,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX14201179""","""China""","""Asia""","[""China: Bengbu""]",null,"""Illumina NovaSeq 6000""","""Berberine2""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,10891,3372,"""pig gut metagenome""","""ILLUMINA""",2022-02-18 00:00:00 UTC,"""SRS12021758""","""SRP360093""",null,null,null,null,"""[""2021-12-10""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa domesticus""]""",null,null,null,null,null,null,null,null,null,"""""Not collected""""",null,null,null,null,"""10891334700""","""3536234220""","""""2022-02-18T08:13:00.000Z""""",null,"""""25980739"""""
"""SRR18048902""","""PRJNA807368""",9,"""Berberine3""",[],"""WGS""",300,"""PRJNA807368""","""SAMN25980740""","[""Metagenome or environmental""]","""ANHUI SCIENCE AND TECHNOLOGY U…",2021-12-10,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX14201180""","""China""","""Asia""","[""China: Bengbu""]",null,"""Illumina NovaSeq 6000""","""Berberine3""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,10629,3288,"""pig gut metagenome""","""ILLUMINA""",2022-02-18 00:00:00 UTC,"""SRS12021759""","""SRP360093""",null,null,null,null,"""[""2021-12-10""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa domesticus""]""",null,null,null,null,null,null,null,null,null,"""""Not collected""""",null,null,null,null,"""10629873300""","""3448076095""","""""2022-02-18T08:14:00.000Z""""",null,"""""25980740"""""
"""SRR18048901""","""PRJNA807368""",9,"

In [42]:
with(pl.Config(tbl_rows=-10)):
    print(metadata_df.filter(pl.col('bioproject') == 'PRJNA807368')['sample_name'].sort())

shape: (12,)
Series: 'sample_name' [str]
[
	"Berberine1"
	"Berberine2"
	"Berberine3"
	"Berberine4"
	"Berberine5"
	"Berberine6"
	"Control1"
	"Control2"
	"Control3"
	"Control4"
	"Control5"
	"Control6"
]


In [43]:
with pl.Config(tbl_rows=-1):
    print(by_acc
     .join(metadata_df, left_on='match_name', right_on='acc', how='inner')
     .filter(pl.col('bioproject') == 'PRJNA807368')
     .select(['n_core', 'match_name', 'sample_name'])
     .sort('n_core')
    )

shape: (12, 3)
┌────────┬─────────────┬─────────────┐
│ n_core ┆ match_name  ┆ sample_name │
│ ---    ┆ ---         ┆ ---         │
│ u32    ┆ str         ┆ str         │
╞════════╪═════════════╪═════════════╡
│ 5      ┆ SRR18048904 ┆ Berberine1  │
│ 8      ┆ SRR18048903 ┆ Berberine2  │
│ 9      ┆ SRR18048901 ┆ Berberine4  │
│ 9      ┆ SRR18048902 ┆ Berberine3  │
│ 10     ┆ SRR18048908 ┆ Control3    │
│ 10     ┆ SRR18048910 ┆ Berberine5  │
│ 10     ┆ SRR18048909 ┆ Berberine6  │
│ 11     ┆ SRR18048911 ┆ Control2    │
│ 13     ┆ SRR18048906 ┆ Control5    │
│ 14     ┆ SRR18048912 ┆ Control1    │
│ 14     ┆ SRR18048907 ┆ Control4    │
│ 17     ┆ SRR18048905 ┆ Control6    │
└────────┴─────────────┴─────────────┘
